In [1]:
import pandas as pd
import json

In [2]:
from google.colab import files
uploaded = files.upload()

Saving watch-history.json to watch-history.json


In [3]:
with open('watch-history.json', 'r', encoding='utf-8') as f:
  data = json.load(f)

In [4]:
print(type(data))
print(data[0])

<class 'list'>
{'header': 'YouTube Music', 'title': 'Watched Smile', 'titleUrl': 'https://music.youtube.com/watch?v=tQTHBlf-Tvw', 'subtitles': [{'name': 'Michael Jackson - Topic', 'url': 'https://www.youtube.com/channel/UCoIOOL7QKuBhQHVKL8y7BEQ'}], 'time': '2026-05-20T15:25:24.427Z', 'products': ['YouTube'], 'activityControls': ['YouTube watch history']}


In [5]:
record = []

for x in data:
  if x.get('header') == 'YouTube Music':
    record.append({
        'title': x.get('title', '').replace('Watched', ''),
        'artist' : x['subtitles'][0]['name'] if x.get('subtitles') else None,
        'url' : x.get('titleUrl'),
        'time': x.get('time')
        })

In [6]:
df = pd.DataFrame(record)
df['time'] = pd.to_datetime(df['time'])

DATA CLEANING

In [7]:
df.isna().sum()

,0
title,0
artist,11
url,0
time,0


In [8]:
df[df['artist'].isna()]

,title,artist,url,time
441,https://music.youtube.com/watch?v=1R4pJc6vNn8,None,https://music.youtube.com/watch?v=1R4pJc6vNn8,2026-04-30 08:34:39.291000+00:00
442,https://music.youtube.com/watch?v=T3FG_YQOW5k,None,https://music.youtube.com/watch?v=T3FG_YQOW5k,2026-04-30 08:34:19.641000+00:00
443,https://music.youtube.com/watch?v=Z85lxckrtzg,None,https://music.youtube.com/watch?v=Z85lxckrtzg,2026-04-30 08:34:07.522000+00:00
444,https://music.youtube.com/watch?v=5b2jFvNKUKc,None,https://music.youtube.com/watch?v=5b2jFvNKUKc,2026-04-30 08:33:38.017000+00:00
445,https://music.youtube.com/watch?v=KFS9c852M_k,None,https://music.youtube.com/watch?v=KFS9c852M_k,2026-04-30 08:33:35.603000+00:00
446,https://music.youtube.com/watch?v=5-jxt7ZyfxE,None,https://music.youtube.com/watch?v=5-jxt7ZyfxE,2026-04-30 08:33:33.187000+00:00
447,https://music.youtube.com/watch?v=HYQN13GZSto,None,https://music.youtube.com/watch?v=HYQN13GZSto,2026-04-30 08:33:31.169000+00:00
448,https://music.youtube.com/watch?v=TC0vu9V8YXw,None,https://music.youtube.com/watch?v=TC0vu9V8YXw,2026-04-30 08:33:29.497000+00:00
449,https://music.youtube.com/watch?v=uHQL9RKh8U8,None,https://music.youtube.com/watch?v=uHQL9RKh8U8,2026-04-30 08:33:19.467000+00:00
450,https://music.youtube.com/watch?v=Qzr0FvjOc3A,None,https://music.youtube.com/watch?v=Qzr0FvjOc3A,2026-04-30 08:29:10.508000+00:00


In [9]:
# drop the null values
df = df.dropna()

In [10]:
# checking for duplicates
df.duplicated().sum()

np.int64(0)

In [11]:
# removed extra spaces from artist
df['artist'] = df['artist'].str.strip()

In [12]:
# removed extra spaces from title column
df['title'] = df['title'].str.strip()

In [13]:
df['artist'] = df['artist'].str.title()

In [14]:
df['title'] = df['title'].str.title()

In [15]:
# converting time zone
df['time'] = df['time'].dt.tz_convert('Asia/Kolkata')

In [16]:
df['date'] = df['time'].dt.date
df['hour'] = df['time'].dt.hour
df['month'] = df['time'].dt.month_name()

In [17]:
df['day_name'] = df['time'].dt.day_name()

In [18]:
df['artist'] = df['artist'].str.replace('- Topic', '', regex = False)

In [35]:
df[df['artist'].str.contains('Michael', na=False)]['artist'].unique()

array(['Michael Jackson ', 'Michaeljacksonvevo'], dtype=object)

In [37]:
df['artist'] = df['artist'].str.strip()
df['artist'] = df['artist'].replace('Michaeljacksonvevo', 'Michael Jackson')

DATA ANALYSIS

In [19]:
# Top 10 artist I streamed songs of the most
df['artist'].value_counts().head(10)

,count
artist,
Michael Jackson,218
Seventeen,17
Shashwat Sachdev,9
Jackson 5,9
Sven Nelson,6
Bts,5
Janet Jackson,4
Bruno Mars,4
Bobby Brown,3


In [20]:
# Top 5 songs
df['title'].value_counts().head(5)

,count
title,
Smooth Criminal (2012 Remaster),13
Billie Jean,11
Don'T Stop 'Til You Get Enough,10
Thriller,10
Beat It,9


In [29]:
df[df['artist'] == 'Michael Jackson']['title'].value_counts().head(5)

,count
title,


In [22]:
# Which hour I listen the most
df['hour'].value_counts().reset_index()

,hour,count
0,19,71
1,21,59
2,11,54
3,18,44
4,22,37
5,10,34
6,20,30
7,13,23
8,23,23
9,9,20


In [23]:
# day I listen the most
df['day_name'].value_counts().reset_index()

,day_name,count
0,Tuesday,101
1,Monday,88
2,Saturday,70
3,Thursday,62
4,Wednesday,59
5,Friday,37
6,Sunday,24


In [24]:
#20 days of may which songs I played the most
df[df['month'] == 'May'][['title', 'artist']].value_counts().head(10)

,,count
title,artist,
Smooth Criminal (2012 Remaster),Michael Jackson,12
Billie Jean,Michael Jackson,10
Don'T Stop 'Til You Get Enough,Michael Jackson,10
Thriller,Michael Jackson,10
Beat It,Michael Jackson,8
Earth Song,Michael Jackson,8
Butterflies,Michael Jackson,8
Dirty Diana (2012 Remaster),Michael Jackson,7
Heaven Can Wait,Michael Jackson,7


In [39]:
mj_count = df[df['artist'] == 'Michael Jackson'].shape[0]
total = df.shape[0]
percentage = round(mj_count/total * 100, 0)

print(f"Mj_songs_played : {mj_count}")
print(f"mj_percent: {percentage}%")

Mj_songs_played : 219
mj_percent: 50.0%


In [25]:
# total songs I listened to in last 20 days
df['title'].nunique()

277

In [ ]:
# day I streamed the most
df['date'].value_counts().head(5)

,count
date,
2026-05-04,56
2026-05-19,45
2026-05-16,41
2026-05-12,40
2026-05-14,32


In [26]:
# saving df to csv

df.to_csv('yt_music_analysis.csv', index=False)

In [40]:
from google.colab import files
files.download('yt_music_analysis.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>